# Multimodal Study Assistant — Pipeline Test Notebook

This notebook walks through ingesting the three sample files (PPTX, PDF, audio) and then querying the resulting vector store.  
Run the cells **top to bottom**. Each section is self-contained so you can re-run individual steps after a fresh ingest.

## 0 · Setup — working directory & imports

In [ ]:
import os, sys
from pathlib import Path

# VS Code injects __vsc_ipynb_file__ with the notebook's absolute path.
# Fall back to cwd() only if running outside VS Code.
if "__vsc_ipynb_file__" in dir():
    PROJ_ROOT = Path(__vsc_ipynb_file__).parent
else:
    PROJ_ROOT = Path.cwd()

os.chdir(PROJ_ROOT)
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))

print(f"Working directory : {Path.cwd()}")
print(f"Python            : {sys.executable}")

Working directory : c:\Users\study\Desktop\FYP_test\multimodal-study-assistant
Python            : c:\Users\study\Desktop\FYP_test\multimodal-study-assistant\venv\Scripts\python.exe


In [ ]:
# Sample file paths — adjust if you've placed your files elsewhere
PPTX_FILE  = "sample_files/CM3060 L6.pptx"
PDF_FILE   = "sample_files/test_notes.pdf"
AUDIO_FILE = "sample_files/test_audio.mp3"

# Every ingest call must supply a session_id.
# Using distinct ids makes it easy to filter queries per file type.
SESSION_PPTX  = "notebook-test-pptx"
SESSION_PDF   = "notebook-test-pdf"
SESSION_AUDIO = "notebook-test-audio"

for p in [PPTX_FILE, PDF_FILE, AUDIO_FILE]:
    exists = Path(p).exists()
    status = "✓ found" if exists else "✗ MISSING"
    print(f"{status}  {p}")

✓ found  sample_files/CM3060 L2.pptx
✓ found  sample_files/test_notes.pdf
✓ found  sample_files/test_audio.mp3


## 1 · Load models (lazy — first call triggers download/load)

In [ ]:
from backend.ingest import ingest_file
from backend.retrieve import query_rag, ask_ollama

print("Imports OK — models will load on first use.")

Imports OK — models will load on first use.


---
## 2 · Ingest — PPTX

Extracts slide titles, body text, speaker notes, and BLIP captions for any embedded images.

In [12]:
import time

t0 = time.perf_counter()
pptx_chunks = ingest_file(PPTX_FILE, {"session_id": SESSION_PPTX})
elapsed = time.perf_counter() - t0

print(f"\n=== PPTX ingest complete ===")
print(f"Chunks stored : {pptx_chunks}")
print(f"Wall time     : {elapsed:.1f}s")


[TIMER] === ingest_file START: CM3060 L2.pptx ===
[TIMER] BLIP caption: 4.98s
[TIMER]   PPTX slide 1: 5.06s
[TIMER] BLIP caption: 2.69s
[TIMER]   PPTX slide 2: 2.71s
[TIMER] BLIP caption: 1.82s
[TIMER]   PPTX slide 3: 1.90s
[TIMER]   PPTX slide 4: 0.00s
[TIMER] BLIP caption: 1.48s
[TIMER]   PPTX slide 5: 1.54s
[TIMER] BLIP caption: 1.91s
[TIMER]   PPTX slide 6: 1.97s
[TIMER]   PPTX slide 7: 0.00s
[TIMER]   PPTX slide 8: 0.00s
[TIMER]   PPTX slide 9: 0.00s
[TIMER]   PPTX slide 10: 0.00s
[TIMER] BLIP caption: 2.09s
[TIMER]   PPTX slide 11: 2.16s
[TIMER]   PPTX slide 12: 0.01s
[TIMER] BLIP caption: 2.36s
[TIMER]   PPTX slide 13: 2.39s
[TIMER]   PPTX slide 14: 0.00s
[TIMER] BLIP caption: 1.69s
[TIMER] BLIP caption: 3.41s
[TIMER]   PPTX slide 15: 5.19s
[TIMER] BLIP caption: 1.84s
[TIMER]   PPTX slide 16: 1.94s
[TIMER]   PPTX slide 17: 0.00s
[TIMER] BLIP caption: 1.48s
[TIMER]   PPTX slide 18: 1.54s
[TIMER] BLIP caption: 1.82s
[TIMER]   PPTX slide 19: 1.88s
[TIMER]   PPTX slide 20: 0.00s
[T

---
## 3 · Ingest — PDF

Text pages are split by paragraph / sliding window.  
Image-heavy pages fall back to BLIP captioning + OCR.

In [ ]:
t0 = time.perf_counter()
pdf_chunks = ingest_file(PDF_FILE, {"session_id": SESSION_PDF})
elapsed = time.perf_counter() - t0

print(f"\n=== PDF ingest complete ===")
print(f"Chunks stored : {pdf_chunks}")
print(f"Wall time     : {elapsed:.1f}s")

---
## 4 · Ingest — Audio (MP3)

Whisper transcribes the audio, then the transcript is split with a 200-word / 50-word-overlap sliding window.

In [ ]:
t0 = time.perf_counter()
audio_chunks = ingest_file(AUDIO_FILE, {"session_id": SESSION_AUDIO})
elapsed = time.perf_counter() - t0

print(f"\n=== Audio ingest complete ===")
print(f"Chunks stored : {audio_chunks}")
print(f"Wall time     : {elapsed:.1f}s")

---
## 5 · Ingest summary

In [ ]:
print("File type | Chunks stored")
print("-" * 30)
print(f"PPTX      | {pptx_chunks}")
print(f"PDF       | {pdf_chunks}")
print(f"Audio     | {audio_chunks}")
print(f"TOTAL     | {pptx_chunks + pdf_chunks + audio_chunks}")

---
## 6 · Query — search within PPTX chunks

`session_id` filters results to only the slides we just ingested.

In [ ]:
PPTX_QUERY = "What are the main topics covered in the slides?"

results = query_rag(PPTX_QUERY, session_id=SESSION_PPTX, n_results=3)

print(f"Query : {PPTX_QUERY}\n")
for i, (chunk, meta, dist) in enumerate(
    zip(results["chunks"], results["metadatas"], results["distances"]), 1
):
    print(f"--- Result {i} (distance={dist:.4f}) ---")
    print(f"  Source : {meta.get('source_file')}  slide {meta.get('page_or_slide')}")
    print(f"  Text   : {chunk[:300]}")
    print()

---
## 7 · Query — search within PDF chunks

In [ ]:
PDF_QUERY = "Summarise the key concepts explained in the notes."

results = query_rag(PDF_QUERY, session_id=SESSION_PDF, n_results=3)

print(f"Query : {PDF_QUERY}\n")
for i, (chunk, meta, dist) in enumerate(
    zip(results["chunks"], results["metadatas"], results["distances"]), 1
):
    print(f"--- Result {i} (distance={dist:.4f}) ---")
    print(f"  Source : {meta.get('source_file')}  page {meta.get('page_or_slide')}")
    print(f"  Text   : {chunk[:300]}")
    print()

---
## 8 · Query — search within Audio transcript chunks

In [ ]:
AUDIO_QUERY = "What does the speaker talk about in the recording?"

results = query_rag(AUDIO_QUERY, session_id=SESSION_AUDIO, n_results=3)

print(f"Query : {AUDIO_QUERY}\n")
for i, (chunk, meta, dist) in enumerate(
    zip(results["chunks"], results["metadatas"], results["distances"]), 1
):
    print(f"--- Result {i} (distance={dist:.4f}) ---")
    print(f"  Source : {meta.get('source_file')}  chunk {meta.get('chunk_index')}")
    print(f"  Text   : {chunk[:300]}")
    print()

---
## 9 · Query — cross-file search (no session filter)

Omitting `session_id` searches across **all** ingested material.

In [ ]:
CROSS_QUERY = "Give me a brief overview of everything that was studied."

results = query_rag(CROSS_QUERY, session_id=None, n_results=5)

print(f"Query : {CROSS_QUERY}\n")
for i, (chunk, meta, dist) in enumerate(
    zip(results["chunks"], results["metadatas"], results["distances"]), 1
):
    print(f"--- Result {i} (distance={dist:.4f}) ---")
    src_type = meta.get('source_type', '?').upper()
    print(f"  [{src_type}] {meta.get('source_file')}")
    print(f"  Text : {chunk[:250]}")
    print()

---
## 10 · RAG answer via Ollama (requires Ollama running locally)

Retrieves the top-5 chunks across all files and feeds them to the local LLM.  
Skipped automatically if Ollama is not reachable.

In [ ]:
import requests

OLLAMA_QUESTION = "Based on the study materials, what are the most important points I should remember?"

# Check if Ollama is running before attempting
try:
    ping = requests.get("http://localhost:11434", timeout=3)
    ollama_ok = ping.status_code == 200
except Exception:
    ollama_ok = False

if not ollama_ok:
    print("Ollama is not running — skipping LLM step.")
    print("Start it with: ollama serve")
else:
    rag_results = query_rag(OLLAMA_QUESTION, session_id=None, n_results=5)
    answer = ask_ollama(OLLAMA_QUESTION, rag_results["chunks"])
    print(f"Question:\n{OLLAMA_QUESTION}\n")
    print(f"Answer:\n{answer}")

---
## 11 · Inspect raw debug files

Ingestion writes every chunk to `chunks_debug_2.json` and every query to `query_debug.json` for easy inspection.

In [ ]:
import json

debug_path = Path("chunks_debug_2.json")
if debug_path.exists():
    data = json.loads(debug_path.read_text(encoding="utf-8"))
    latest = data[-1]  # Most recent ingest batch
    print(f"Latest ingest batch saved at : {latest['saved_at']}")
    print(f"Number of chunks in batch    : {len(latest['chunks'])}")
    print("\nFirst chunk preview:")
    first = latest["chunks"][0]
    print(f"  metadata : {first['metadata']}")
    print(f"  text     : {first['text'][:300]}")
else:
    print("chunks_debug_2.json not found — run the ingest cells first.")

In [ ]:
query_debug_path = Path("query_debug.json")
if query_debug_path.exists():
    qdata = json.loads(query_debug_path.read_text(encoding="utf-8"))
    latest_q = qdata[-1]
    print(f"Latest query asked at : {latest_q['asked_at']}")
    print(f"Question              : {latest_q['question']}")
    print(f"Chunks retrieved      : {len(latest_q['retrieved'])}")
    print("\nTop result:")
    top = latest_q["retrieved"][0]
    print(f"  rank     : {top['rank']}")
    print(f"  distance : {top['distance']:.4f}")
    print(f"  metadata : {top['metadata']}")
    print(f"  text     : {top['text'][:300]}")
else:
    print("query_debug.json not found — run the query cells first.")

---
## 12 · OCR-first fix — single slide inspection

Targets slide 5 ("Ambiguous And Complex") where the bullet-point content is baked into an image.  
Runs both OCR and BLIP on the embedded image so you can compare the old vs new output.

In [17]:
from spire.presentation import Presentation as SpirePresentation, FileFormat

SLIDE_INDEX = 1  # 0-based → change this to test any slide

spire_prs = SpirePresentation()
spire_prs.LoadFromFile(PPTX_FILE)

slide = spire_prs.Slides[SLIDE_INDEX]
print(f"=== Slide {SLIDE_INDEX + 1} — Spire text extraction ===\n")

all_text = []
for shape in slide.Shapes:
    try:
        text = shape.TextFrame.Text.strip()
        if text:
            print(f"  Shape [{shape.Name}]: {repr(text[:200])}")
            all_text.append(text)
    except Exception:
        pass  # shape has no TextFrame (images, charts, etc.)

spire_prs.Dispose()

print(f"\n=== All text found ({len(all_text)} shapes with text) ===\n")
print("\n---\n".join(all_text))

=== Slide 2 — Spire text extraction ===

  Shape [Title 1]: 'Topic 5: Text Categorization'

=== All text found (1 shapes with text) ===

Topic 5: Text Categorization
